In [ ]:
import import_ipynb
from Modelo import Civil, Impostor, PistaInvalidaException
from abc import ABC, abstractmethod
import random
class IAsignadorRoles(ABC):
    @abstractmethod
    def asignar(self, jugadores, palabra_secreta):
        pass

class AsignadorRolesAleatorio(IAsignadorRoles):
    def asignar(self, jugadores, palabra_secreta):
        if not 3 <= len(jugadores) <= 10:
            raise ValueError("Debe haber minimo 3 jugadores y  máximo 10 jugadores")
        impostor = random.choice(jugadores)
        for i in jugadores:
            if i is impostor:
                i.set_rol(Impostor())
            else:
                i.set_rol(Civil(palabra_secreta))


class IValidadorPista(ABC):
    @abstractmethod
    def validar(self, pista):
        pass
class ValidadorPalabraProhibida(IValidadorPista):
    def __init__(self, palabra_secreta):
        self._palabra_secreta = palabra_secreta
    def validar(self, pista):
        if self._palabra_secreta.texto.lower() in pista.get_texto().lower():
            raise PistaInvalidaException("Incorrecto, la pista no puede tener la palabra secreta")

class ValidadorLongitud(IValidadorPista):
    def __init__(self, longitud_maxima):
        self._longitud_maxima = longitud_maxima

    def validar(self, pista):
        if len(pista.get_texto()) > self._longitud_maxima:
            raise PistaInvalidaException(f"La pista no puede tener más de {self._longitud_maxima} caracteres.")


class IContadorVotos(ABC):
    @abstractmethod
    def contar(self, votos):
        pass

class ContadorVotosMayoria(IContadorVotos):
    def contar(self, votos):
        votos_por_acusado = {}
        for voto in votos:
            acusado = voto.get_acusado()
            votos_por_acusado[acusado] = votos_por_acusado.get(acusado, 0) + 1
        if not votos_por_acusado:
            return None
        maximo = max(votos_por_acusado.values())
        participantes_mas_votados = [a for a, num_votos in votos_por_acusado.items() if num_votos == maximo]
        if len(participantes_mas_votados) > 1:
            return None
        return participantes_mas_votados[0]


class IRepositorioPalabras(ABC):
    @abstractmethod
    def obtener(self, categoria): pass

    @abstractmethod
    def obtener_palabra_aleatoria(self, categoria): pass


class RepositorioPalabrasMemoria(IRepositorioPalabras):
    def __init__(self):
        datos = {
            "Comida y bebidas": ["Sushi", "Tacos", "Hamburguesa", "Paella", "Ceviche",
              "Helado", "Cafe", "Chocolate", "Empanada", "Lasaña", "Croissant",
              "Palomitas de maiz", "Aguacate", "Queso", "Malteada"],
            "Animales": ["Leon", "Pingüino", "Camaleon", "Delfin", "Murcielago", "Pulpo",
              "Jirafa", "Escorpion", "Koala", "Flamenco", "Cocodrilo",
              "Abeja", "Lobo", "Ornitorrinco", "Buho", "Perro"],
            "Lugares": ["Aeropuerto", "Casino", "Submarino", "Zoologico", "Cementerio",
              "Gimnasio", "Biblioteca", "Cine", "Crucero", "Estacion espacial",
              "Circo", "Supermercado", "Playa", "Carcel"],
            "Profesiones y oficios": ["Bombero", "Astronauta", "Detective", "Cirujano", "Mago",
              "Arqueologo", "Chef", "Piloto de avión", "Payaso", "Juez",
              "Fotografo", "Buzo", "Mecanico", "Carpintero", "Espia", "Ingeniero"],
            "Objetos cotidianos": ["Paraguas", "Espejo", "Reloj de arena", "Llave", "Tijeras",
              "Linterna", "Almohada", "Brujula", "Camara", "Vela",
              "Mochila", "Cepillo de dientes", "Candado", "Control remoto", "Iman"],
            "Cine, fantasía y cultura pop": ["Vampiro", "Zombi", "Superheroe", "Pirata", "Extraterrestre",
              "Fantasma", "Sirena", "Dragon", "Ninja", "Caballero",
              "Robot", "Momia", "Bruja", "Hombre lobo", "Dinosaurio"],
        }
        self._categorias = {}
        for nombre, palabras in datos.items():
            categoria = Categoria(nombre)
            for palabra in palabras:
                categoria.agregar_palabra(palabra)
            self._categorias[nombre.lower()] = categoria

    def obtener(self, categoria):
        clave = categoria.strip().lower()
        if clave not in self._categorias:
            raise CategoriaInvalidaException(f"La categoría '{categoria}' no existe.")
        return self._categorias[clave]
    
    def listar_categorias(self):
      return [c.tipo for c in self._categorias.values()]

    def obtener_palabra_aleatoria(self, categoria):
        categva = self.obtener(categoria)
        return PalabraSecreta(random.choice(categva.palabras), categva)


In [ ]:
import unittest


class TestAsignadorRoles(unittest.TestCase):
    def setUp(self):
        self.palabra = PalabraSecreta("Empanada", Categoria("Comida y bebidas"))
        self.jugadores = [Jugador(i, f"Jugador {i}") for i in range(7)]
        AsignadorRolesAleatorio().asignar(self.jugadores, self.palabra)

    def test_un_solo_impostor(self):
        impostores = [j for j in self.jugadores if isinstance(j.get_rol(), Impostor)]
        self.assertEqual(len(impostores), 1)

    def test_demas_jugadores_son_civiles(self):
        civiles = [j for j in self.jugadores if isinstance(j.get_rol(), Civil)]
        self.assertEqual(len(civiles), 6)

    def test_civil_ve_la_palabra_e_impostor_no(self):
        for j in self.jugadores:
            if isinstance(j.get_rol(), Civil):
                self.assertEqual(j.get_rol().obtener_palabra_visible(), "Empanada")
            else:
                self.assertEqual(j.get_rol().obtener_palabra_visible(), "???")

    def test_menos_de_3_jugadores(self):
        with self.assertRaises(ValueError):
            AsignadorRolesAleatorio().asignar(self.jugadores[:2], self.palabra)

    def test_mas_de_10_jugadores(self):
        once = [Jugador(i, f"J{i}") for i in range(11)]
        with self.assertRaises(ValueError):
            AsignadorRolesAleatorio().asignar(once, self.palabra)


class TestValidacionPistas(unittest.TestCase):
    def setUp(self):
        self.palabra = PalabraSecreta("Playa", Categoria("Lugares"))
        self.jugador = Jugador(1, "Ana")

    def test_rechaza_palabra_secreta(self):
        pista = self.jugador.dar_pista("Me gusta la playa", 1)
        with self.assertRaises(PistaInvalidaException):
            ValidadorPalabraProhibida(self.palabra).validar(pista)

    def test_rechaza_pista_muy_larga(self):
        pista = self.jugador.dar_pista("una pista demasiado larga", 1)
        with self.assertRaises(PistaInvalidaException):
            ValidadorLongitud(10).validar(pista)

    def test_acepta_pista_valida(self):
        pista = self.jugador.dar_pista("Arena", 1)
        ValidadorPalabraProhibida(self.palabra).validar(pista)
        ValidadorLongitud(10).validar(pista)


class TestConteoVotos(unittest.TestCase):
    def setUp(self):
        self.a, self.b, self.c, self.d = (Jugador(i, n) for i, n in enumerate("ABCD"))
        self.contador = ContadorVotosMayoria()

    def test_gana_el_mas_votado(self):
        votos = [self.a.votar(self.c), self.b.votar(self.c), self.d.votar(self.a)]
        self.assertIs(self.contador.contar(votos), self.c)

    def test_voto_a_si_mismo(self):
        with self.assertRaises(VotoInvalidoException):
            self.a.votar(self.a)

    def test_empate_devuelve_none(self):
        votos = [self.a.votar(self.b), self.c.votar(self.d)]
        self.assertIsNone(self.contador.contar(votos))


unittest.main(argv=[''], exit=False)